## 05_2. Why first-season members get their own model

In [ ]:
# 2.1 Cancel rate (%) by tenure, before and after the cut
by_tenure = (100 * d.pivot_table(index='post', columns='tenure_group', values='cancel')).round(1)
display(by_tenure.rename(index={0: 'pre-cut', 1: 'post-cut'}).rename_axis(index=None, columns='seasons as a member (8 = 8 or more)'))

# 2.2 First-season members: cancel rate (%) by the price they joined at, at each price charged this season
first = d[d['first'].eq(1)].copy()
later = d[d['first'].eq(0)].copy()
first['joined'] = np.where(first['t'].gt(CUT), 'joined post-cut', 'joined pre-cut')
display((100 * first.pivot_table(index=['joined', 'join_price'], columns='price', values='cancel')).round(1))

# 2.3 One pooled model vs one model per group: which reproduces the cancel rate at each price?
pooled = u.fit_logit(f'cancel ~ price_c + price_sq + C(tenure_group) + {FE}', d)
check = d[['first', 'post', 'price', 'cancel']].assign(pooled=pooled.predict(d), separate=np.nan)
for flag, g, rhs in [(1, first, ''), (0, later, 'C(tenure_group) + ')]:
    check.loc[check['first'].eq(flag), 'separate'] = u.fit_logit(f'cancel ~ price_c + price_sq + {rhs}{FE}', g).predict(g).to_numpy()
check = 100 * (check.assign(group=np.where(check['first'].eq(1), 'first-season', 'later'),
                            period=np.where(check['post'].eq(1), 'post-cut', 'pre-cut'))
                    .groupby(['group', 'period', 'price'])[['cancel', 'pooled', 'separate']].mean())
check['pooled gap pp'] = check['cancel'] - check['pooled']
check['separate gap pp'] = check['cancel'] - check['separate']
print(f"Largest gap at any price: {check['pooled gap pp'].abs().max():.1f} pp pooled, "
      f"{check['separate gap pp'].abs().max():.1f} pp separate")
del pooled
(check.round(1).reset_index()
      .rename(columns={'cancel': 'actual %', 'pooled': 'pooled model %', 'separate': 'separate models %'}))

seasons as a member (8 = 8 or more),1,2,3,4,5,6,7,8
pre-cut,22.2,16.9,20.1,19.8,16.4,14.7,12.7,15.4
post-cut,15.9,10.1,5.0,4.6,3.8,7.8,6.7,6.1


price                         7     8     9     10    11    12    13    14
joined          join_price                                                
joined post-cut 7           17.1  19.9  25.9  27.5  33.7   NaN   NaN   NaN
                8           17.1  21.4  25.0  28.1  32.3   NaN   NaN   NaN
                9           14.0  19.8  23.1  23.4  29.8   NaN   NaN   NaN
                10          14.5  17.1  22.8  23.5  31.1   NaN   NaN   NaN
                11          10.5  15.8  16.8  21.5  31.3   NaN   NaN   NaN
                12          10.4  12.9  15.7  19.9  23.8   NaN   NaN   NaN
joined pre-cut  10           7.2   9.2  10.8  14.4  19.5  23.8  28.1  33.0
                11           8.0   8.2   9.9  13.7  16.5  20.7  26.8  30.9
                12           6.5   8.0  10.7  13.5  16.6  19.6  26.7  32.2
                13           3.8   9.1   7.8  13.2  15.3  20.3  25.1  31.1
                14           8.0   8.5  11.4  12.5  17.0  21.5  27.8  30.2
                15           5.5   7.2   8.5  13.0  16.0  20.7  26.1  32.8

Largest gap at any price: 5.1 pp pooled, 0.4 pp separate


,group,period,price,actual %,pooled model %,separate models %,pooled gap pp,separate gap pp
0,first-season,post-cut,7,10.3,7.4,10.5,2.9,-0.2
1,first-season,post-cut,8,12.9,9.0,12.6,3.9,0.3
2,first-season,post-cut,9,15.7,11.1,15.3,4.6,0.4
3,first-season,post-cut,10,18.4,13.8,18.7,4.6,-0.3
4,first-season,post-cut,11,22.4,17.3,22.6,5.1,-0.2
5,first-season,pre-cut,10,13.7,15.0,13.9,-1.3,-0.1
6,first-season,pre-cut,11,17.3,18.7,17.2,-1.4,0.1
7,first-season,pre-cut,12,21.2,23.3,21.3,-2.1,-0.1
8,first-season,pre-cut,13,26.7,28.9,26.3,-2.3,0.4
9,first-season,pre-cut,14,32.2,35.8,32.4,-3.6,-0.2


**Reading**

* **First-season members cancel the most.** 22% before the cut and 16% after, against 4–10% for later members after the cut.
* **One model for everyone gets first-season members wrong.** A pooled model with a tenure dummy, but one price curve and one set of season effects, under-predicts first-season cancelling after the cut by 2.9–5.1 pp and over-predicts later members by 0.9–1.5 pp. Before the cut the errors run the other way (up to 3.6 pp). The cut lowered later members' cancelling far more than first-season members', and shared season effects cannot carry both. Fitted separately, both groups are within 0.4 pp at every price.
* **For them, last season's price means something different.** It is the price they joined at, which says who they are: after the cut, members who joined at \$7 cancel more at every price than those who joined at \$12 (25.9% against 15.7% when charged \$9). For later members it is just a price they paid.